# Eval full 11.089 — P0 stage 1 FP32 5 epoch

Đánh giá checkpoint `final` của run full 44.331 train, FP32, 5 epoch.

- Không train lại.
- Full validation 11.089 dòng, `target`, beam 4, FP32.
- Đây là **partial probe 5 epoch**, không phải 50 epoch hay paper-final.


In [ ]:
!nvidia-smi
import torch
print("torch:", torch.__version__)
print("cuda devices:", torch.cuda.device_count())


In [ ]:
!rm -rf /kaggle/working/Vuln2Action-Lab
!git clone --branch feature/paper-reproduction https://github.com/Hoang-Minh-Hieu-2806/Vuln2Action-Lab.git /kaggle/working/Vuln2Action-Lab
%cd /kaggle/working/Vuln2Action-Lab
!pip install -q -e ".[train]"


In [ ]:
from pathlib import Path
import glob, os
from transformers import AutoTokenizer

hits = sorted(glob.glob("/kaggle/input/**/flow_validation.jsonl", recursive=True))
print("flow_validation hits:", hits)
if not hits:
    raise FileNotFoundError("flow_validation.jsonl not found")
DATA_DIR = Path(hits[0]).parent

model_hits = sorted(glob.glob("/kaggle/input/vuln2action-p0-stage1-fp32-5ep/config.json"))
print("model final hits:", model_hits)
if not model_hits:
    raise FileNotFoundError("config.json not found")
CHECKPOINT = Path(model_hits[0]).parent

os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("DATA_DIR:", DATA_DIR)
print("CHECKPOINT:", CHECKPOINT)
tok = AutoTokenizer.from_pretrained(str(CHECKPOINT))
print("tokenizer:", type(tok).__name__, "vocab", len(tok))


In [ ]:
from pathlib import Path
import subprocess, sys, os

cmd = [
    sys.executable, "scripts/evaluate_t5.py",
    "--checkpoint", str(CHECKPOINT),
    "--dataset", str(DATA_DIR),
    "--split", "validation",
    "--target-field", "target",
    "--output-dir", "/kaggle/working/eval_5ep_full",
    "--max-input-length", "512",
    "--max-target-length", "512",
    "--batch-size", "4",
    "--num-beams", "4",
    "--no-fp16",
]
print(" ".join(cmd))
subprocess.run(cmd, cwd="/kaggle/working/Vuln2Action-Lab", check=True)


In [ ]:
import json
from pathlib import Path

out = Path("/kaggle/working/eval_5ep_full")
m = json.loads((out / "metrics.json").read_text(encoding="utf-8"))
print(json.dumps(m, indent=2, ensure_ascii=False))

rows = [json.loads(line) for line in (out / "predictions.jsonl").read_text(encoding="utf-8").splitlines()]
empty = sum(1 for r in rows if not r.get("prediction", "").strip())
print(f"empty predictions: {empty}/{len(rows)}")
print("rows:", len(rows))
for i, r in enumerate(rows[:3]):
    print(f"\n[{i}] {r.get('cve_id')}")
    print("REF:", (r.get("reference") or "")[:400])
    print("PRED:", (r.get("prediction") or "")[:400])


## Cách đọc

- BLEU/ROUGE ở đây là **5 epoch**, full validation 11.089 dòng.
- Không so trực tiếp với BLEU `0,7589` của paper nếu chưa đủ 50 epoch và cùng
  dataset gốc.
- Nếu prediction rỗng hoặc BLEU rất thấp: kiểm tra checkpoint, target length,
  và lỗi repetition trước khi kết luận.
